# Weather Join (Flights ✕ Weather)\n\nJoin processed hourly weather observations to cleaned BTS flights using **nearest-hour matching** (`merge_asof`), then derive weather-related features.\n\n### Strategy\n1. **Interpolate** weather gaps (linear interpolation + forward/backward fill) to reduce missingness before joining.\n2. Round each flight's `CRSDepTime` / `CRSArrTime` to the **nearest whole hour** and match to the closest weather observation within a 2-hour tolerance.\n3. Origin weather is matched on **departure time**; destination weather on **scheduled arrival time**.\n4. Derive binary flags (rain, high wind, freezing, low visibility) and a composite severity score.\n\n### Inputs\n- `data/processed/bts/flights_2024_clean.parquet`\n- `data/processed/weather/weather_2024_hourly.parquet`\n\n### Outputs\n- `data/processed/integrated/flights_2024_weather.parquet`\n- `data/reports/integrated/weather_join_quality_2024.csv`

## Step 0: Ensure project data (local or Hugging Face)\n\nDownloads [ahiruuu/CIS-5450](https://huggingface.co/datasets/ahiruuu/CIS-5450) when `data/` is missing. Override with `HF_DATA_REPO_ID` if needed.

In [1]:
import sys
from pathlib import Path

_here = Path.cwd().resolve()
for _p in [_here] + list(_here.parents):
    if (_p / "notebooks" / "project_data.py").exists():
        sys.path.insert(0, str(_p / "notebooks"))
        break

from project_data import ensure_project_data

ensure_project_data()

[data] Already present: C:\Users\18724\Desktop\CIS5450\teamwork\data


WindowsPath('C:/Users/18724/Desktop/CIS5450/teamwork/data')

In [2]:
## Step 1: Environment setup and data loading

In [3]:
import os, warnings
from pathlib import Path

import numpy as np
import pandas as pd

from project_data import resolve_project_root

warnings.filterwarnings("ignore", category=FutureWarning)

PROJECT_ROOT = resolve_project_root()
DATA_ROOT = Path(os.getenv("FLIGHT_DATA_DIR", PROJECT_ROOT / "data")).expanduser().resolve()
PROCESSED_BTS_DIR = DATA_ROOT / "processed" / "bts"
PROCESSED_WEATHER_DIR = DATA_ROOT / "processed" / "weather"
PROCESSED_INTEGRATED_DIR = DATA_ROOT / "processed" / "integrated"
REPORT_DIR = DATA_ROOT / "reports" / "integrated"
PROCESSED_INTEGRATED_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

flights_path = PROCESSED_BTS_DIR / "flights_2024_clean.parquet"
weather_path = PROCESSED_WEATHER_DIR / "weather_2024_hourly.parquet"

if not flights_path.exists():
    raise FileNotFoundError(f"Missing input: {flights_path}")
if not weather_path.exists():
    raise FileNotFoundError(f"Missing input: {weather_path}")

flights = pd.read_parquet(flights_path)
weather = pd.read_parquet(weather_path)

print(f"Project root:  {PROJECT_ROOT}")
print(f"Flights rows:  {len(flights):,}")
print(f"Weather rows:  {len(weather):,}")
print(f"Weather airports: {weather['IATA'].nunique()}")

Project root:  C:\Users\18724\Desktop\CIS5450\teamwork
Flights rows:  6,817,598
Weather rows:  438,299
Weather airports: 50


In [4]:
## Step 2: Interpolate weather missing values\n\nApply **linear interpolation** (gap ≤ 6 hours) within each airport, then forward-fill and backward-fill remaining edge values (limit 3 hours). This reduces missingness before the join step.

In [5]:
weather_cols = [
    "air_temp", "dew_point", "sea_level_pres", "wind_dir",
    "wind_speed", "sky_cover", "precip_1h", "precip_6h",
]

print("Weather missingness BEFORE interpolation:")
before_miss = {}
for c in weather_cols:
    pct = weather[c].isna().mean() * 100
    before_miss[c] = pct
    print(f"  {c:20s}: {pct:.2f}%")

weather["obs_time"] = weather["obs_time"].astype("datetime64[ns]")
weather = weather.sort_values(["IATA", "obs_time"]).reset_index(drop=True)

# Linear interpolation within each airport (gap limit = 6 hours)
weather[weather_cols] = (
    weather.groupby("IATA")[weather_cols]
    .transform(lambda s: s.interpolate(method="linear", limit=6))
)
# Forward-fill remaining gaps at series edges
weather[weather_cols] = (
    weather.groupby("IATA")[weather_cols]
    .transform(lambda s: s.ffill(limit=3))
)
# Backward-fill leading NaNs
weather[weather_cols] = (
    weather.groupby("IATA")[weather_cols]
    .transform(lambda s: s.bfill(limit=3))
)

print("\nWeather missingness AFTER interpolation:")
for c in weather_cols:
    pct = weather[c].isna().mean() * 100
    print(f"  {c:20s}: {before_miss[c]:.2f}% -> {pct:.2f}%")

Weather missingness BEFORE interpolation:
  air_temp            : 0.00%
  dew_point           : 0.00%
  sea_level_pres      : 0.32%
  wind_dir            : 0.05%
  wind_speed          : 0.00%
  sky_cover           : 7.53%
  precip_1h           : 4.21%
  precip_6h           : 86.73%

Weather missingness AFTER interpolation:
  air_temp            : 0.00% -> 0.00%
  dew_point           : 0.00% -> 0.00%
  sea_level_pres      : 0.32% -> 0.21%
  wind_dir            : 0.05% -> 0.00%
  wind_speed          : 0.00% -> 0.00%
  sky_cover           : 7.53% -> 3.31%
  precip_1h           : 4.21% -> 3.04%
  precip_6h           : 86.73% -> 68.24%


## Step 3: Prepare flight datetime columns\n\nRound `CRSDepTime` (HHMM) to the **nearest whole hour** to build `dep_datetime`. Similarly, round `CRSArrTime` to build `arr_datetime` (used for destination weather). Overnight flights (arrival hour ≤ departure hour) are shifted forward by one day.

In [6]:
flights["FlightDate"] = pd.to_datetime(flights["FlightDate"], errors="coerce")
flights["Origin"] = flights["Origin"].astype(str)
flights["Dest"] = flights["Dest"].astype(str)

initial_rows = len(flights)

# CRSDepTime -> departure datetime (rounded to nearest hour)
dep_hhmm = pd.to_numeric(flights["CRSDepTime"], errors="coerce")
dep_hh = (dep_hhmm // 100).clip(0, 23)
dep_mm = dep_hhmm % 100
dep_hour_rounded = dep_hh + (dep_mm >= 30).astype(int)
dep_hour_rounded = dep_hour_rounded.clip(0, 23)
flights["dep_datetime"] = flights["FlightDate"] + pd.to_timedelta(dep_hour_rounded, unit="h")

# CRSArrTime -> arrival datetime (rounded to nearest hour)
arr_hhmm = pd.to_numeric(flights["CRSArrTime"], errors="coerce")
arr_hh = (arr_hhmm // 100).clip(0, 23)
arr_mm = arr_hhmm % 100
arr_hour_rounded = arr_hh + (arr_mm >= 30).astype(int)
arr_hour_rounded = arr_hour_rounded.clip(0, 23)
flights["arr_datetime"] = flights["FlightDate"] + pd.to_timedelta(arr_hour_rounded, unit="h")

# Unify datetime resolution for merge_asof
flights["dep_datetime"] = flights["dep_datetime"].astype("datetime64[ns]")
flights["arr_datetime"] = flights["arr_datetime"].astype("datetime64[ns]")

# Overnight flights: arrival appears earlier than departure -> add 1 day
overnight = flights["arr_datetime"] <= flights["dep_datetime"]
flights.loc[overnight, "arr_datetime"] += pd.Timedelta(days=1)

print(f"Departure datetime range: {flights['dep_datetime'].min()} ~ {flights['dep_datetime'].max()}")
print(f"Arrival datetime range:   {flights['arr_datetime'].min()} ~ {flights['arr_datetime'].max()}")
print(f"Overnight flights adjusted: {overnight.sum():,}")

Departure datetime range: 2024-01-01 00:00:00 ~ 2024-12-31 23:00:00
Arrival datetime range:   2024-01-01 03:00:00 ~ 2025-01-01 23:00:00
Overnight flights adjusted: 413,685


## Step 4: Join origin weather via `merge_asof`\n\nMatch each flight's `Origin` + `dep_datetime` to the **nearest** hourly weather observation at that airport. A 2-hour tolerance ensures we don't match distant observations.

In [7]:
origin_wx = weather[["IATA", "obs_time"] + weather_cols].copy()
origin_wx = origin_wx.sort_values("obs_time").reset_index(drop=True)
origin_rename = {c: f"origin_{c}" for c in weather_cols}
origin_wx = origin_wx.rename(columns={**origin_rename, "IATA": "Origin", "obs_time": "dep_datetime"})

flights = flights.sort_values("dep_datetime").reset_index(drop=True)

flights = pd.merge_asof(
    flights,
    origin_wx,
    on="dep_datetime",
    by="Origin",
    direction="nearest",
    tolerance=pd.Timedelta(hours=2),
)

origin_match_rate = 1 - flights["origin_air_temp"].isna().mean()
print(f"Origin weather match rate: {origin_match_rate * 100:.2f}%")
print(f"Rows after origin join: {len(flights):,} (should equal {initial_rows:,})")

Origin weather match rate: 79.52%
Rows after origin join: 6,817,598 (should equal 6,817,598)


## Step 5: Join destination weather via `merge_asof`\n\nMatch each flight's `Dest` + `arr_datetime` to the nearest hourly weather observation at the destination airport (same 2-hour tolerance).

In [8]:
dest_wx = weather[["IATA", "obs_time"] + weather_cols].copy()
dest_wx = dest_wx.sort_values("obs_time").reset_index(drop=True)
dest_rename = {c: f"dest_{c}" for c in weather_cols}
dest_wx = dest_wx.rename(columns={**dest_rename, "IATA": "Dest", "obs_time": "arr_datetime"})

flights = flights.sort_values("arr_datetime").reset_index(drop=True)

flights = pd.merge_asof(
    flights,
    dest_wx,
    on="arr_datetime",
    by="Dest",
    direction="nearest",
    tolerance=pd.Timedelta(hours=2),
)

dest_match_rate = 1 - flights["dest_air_temp"].isna().mean()
print(f"Dest weather match rate: {dest_match_rate * 100:.2f}%")
print(f"Rows after dest join: {len(flights):,} (should equal {initial_rows:,})")

Dest weather match rate: 79.49%
Rows after dest join: 6,817,598 (should equal 6,817,598)


## Step 6: Derive weather features\n\nCreate binary indicator flags and a composite severity score for both origin and destination weather. Also compute worst-case metrics across the two endpoints.

In [9]:
# Origin binary flags
flights["origin_is_rain"]    = (flights["origin_precip_1h"] > 0).astype(int)
flights["origin_high_wind"]  = (flights["origin_wind_speed"] > 10).astype(int)
flights["origin_freezing"]   = (flights["origin_air_temp"] < 0).astype(int)
flights["origin_low_vis"]    = (flights["origin_sky_cover"] >= 7).astype(int)

flights["origin_weather_severity"] = (
    flights["origin_is_rain"] * 1.0
    + flights["origin_high_wind"] * 1.5
    + flights["origin_freezing"] * 1.2
    + flights["origin_low_vis"] * 0.8
)

# Destination binary flags
flights["dest_is_rain"]    = (flights["dest_precip_1h"] > 0).astype(int)
flights["dest_high_wind"]  = (flights["dest_wind_speed"] > 10).astype(int)
flights["dest_freezing"]   = (flights["dest_air_temp"] < 0).astype(int)
flights["dest_low_vis"]    = (flights["dest_sky_cover"] >= 7).astype(int)

flights["dest_weather_severity"] = (
    flights["dest_is_rain"] * 1.0
    + flights["dest_high_wind"] * 1.5
    + flights["dest_freezing"] * 1.2
    + flights["dest_low_vis"] * 0.8
)

# Worst-case across origin and destination
flights["worst_precip"] = flights[["origin_precip_1h", "dest_precip_1h"]].max(axis=1)
flights["worst_wind"]   = flights[["origin_wind_speed", "dest_wind_speed"]].max(axis=1)

# Drop helper datetime columns used only for joining
flights = flights.drop(columns=["dep_datetime", "arr_datetime"], errors="ignore")

print(f"Derived features added. Total columns: {len(flights.columns)}")
print(f"\nOrigin severity distribution:")
print(flights["origin_weather_severity"].describe())
print(f"\nDest severity distribution:")
print(flights["dest_weather_severity"].describe())

Derived features added. Total columns: 58

Origin severity distribution:
count    6.817598e+06
mean     2.567795e-01
std      5.497052e-01
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      4.500000e+00
Name: origin_weather_severity, dtype: float64

Dest severity distribution:
count    6.817598e+06
mean     2.529106e-01
std      5.462312e-01
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      4.500000e+00
Name: dest_weather_severity, dtype: float64


## Step 7: Quality report and save outputs\n\nVerify row count is preserved (left join should not add or drop rows), summarize join quality, and write the integrated parquet file.

In [10]:
origin_missing_pct = round(flights["origin_air_temp"].isna().mean() * 100, 4)
dest_missing_pct   = round(flights["dest_air_temp"].isna().mean() * 100, 4)

print("=" * 60)
print("WEATHER JOIN QUALITY REPORT")
print("=" * 60)
print(f"Final rows:              {len(flights):,}")
print(f"Final columns:           {len(flights.columns)}")
print(f"Row count preserved:     {len(flights) == initial_rows}")
print(f"Origin weather missing:  {origin_missing_pct:.2f}%")
print(f"Dest weather missing:    {dest_missing_pct:.2f}%")
print(f"Origin match rate:       {origin_match_rate * 100:.2f}%")
print(f"Dest match rate:         {dest_match_rate * 100:.2f}%")

# Note: ~20% missing is expected because the cleaned flights table includes
# routes to/from airports outside the top-50 that have weather station data.
# Those flights retain NaN for weather columns.

# Save integrated parquet
out_path = PROCESSED_INTEGRATED_DIR / "flights_2024_weather.parquet"
flights.to_parquet(out_path, index=False)

# Save quality CSV
quality = pd.DataFrame([{
    "rows": int(len(flights)),
    "columns": int(len(flights.columns)),
    "origin_air_temp_missing_pct": origin_missing_pct,
    "dest_air_temp_missing_pct": dest_missing_pct,
    "origin_match_rate_pct": round(origin_match_rate * 100, 4),
    "dest_match_rate_pct": round(dest_match_rate * 100, 4),
}])
quality_path = REPORT_DIR / "weather_join_quality_2024.csv"
quality.to_csv(quality_path, index=False)

print(f"\nSaved integrated file:  {out_path}")
print(f"Saved quality report:   {quality_path}")
print(f"File size: {out_path.stat().st_size / 1024 / 1024:.1f} MB")
print("=" * 60)

WEATHER JOIN QUALITY REPORT
Final rows:              6,817,598
Final columns:           58
Row count preserved:     True
Origin weather missing:  20.48%
Dest weather missing:    20.51%
Origin match rate:       79.52%
Dest match rate:         79.49%

Saved integrated file:  C:\Users\18724\Desktop\CIS5450\teamwork\data\processed\integrated\flights_2024_weather.parquet
Saved quality report:   C:\Users\18724\Desktop\CIS5450\teamwork\data\reports\integrated\weather_join_quality_2024.csv
File size: 248.8 MB
